# WiDS 2026 — Near-Zone Time Expert Submission

In [1]:
import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import norm
from sklearn.base import clone
from sklearn.linear_model import Ridge
from sklearn.model_selection import KFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR

warnings.filterwarnings("ignore")

SEEDS = tuple(range(10))
N_SPLITS = 5
BLEND_SVR = 0.80
BLEND_RIDGE = 0.20
SIGMA_12 = 1.15
SIGMA_24 = 0.40
SIGMA_48 = 0.94
THRESHOLD_M = 5000.0


def locate_data_dir() -> Path:
    candidates = [
        Path("/kaggle/input/competitions/WiDSWorldWide_GlobalDathon26"),
        Path("/kaggle/input/widsworldwide-globaldathon26"),
        Path("/kaggle/input/WiDSWorldWide_GlobalDathon26"),
        Path("/mnt/data"),
    ]
    for p in candidates:
        if (p / "train.csv").exists() and (p / "test.csv").exists():
            return p
    root = Path("/kaggle/input")
    if root.exists():
        for p in root.glob("**/train.csv"):
            parent = p.parent
            if (parent / "test.csv").exists() and (parent / "sample_submission.csv").exists():
                return parent
    raise FileNotFoundError("Could not find train.csv / test.csv / sample_submission.csv")


def create_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    dist = out["dist_min_ci_0_5h"].clip(lower=1.0)
    speed = out["closing_speed_m_per_h"]
    speed_pos = speed.clip(lower=0.0)
    radial = out["radial_growth_rate_m_per_h"].clip(lower=0.0)
    effective = speed_pos + radial
    per = out["num_perimeters_0_5h"]
    area_first = out["area_first_ha"]

    out["log_distance"] = np.log1p(dist)
    out["dist_km"] = dist / 1000.0
    out["log_eta"] = np.log1p(np.where(speed_pos > 0.01, dist / speed_pos, 9999.0).clip(max=9999.0))
    out["eta_effective"] = np.where(effective > 0.01, dist / effective, 9999.0).clip(max=9999.0)
    out["threat_score"] = out["alignment_abs"] * speed / np.log1p(dist)
    out["fire_urgency"] = per * speed
    out["growth_intensity"] = out["area_growth_rate_ha_per_h"] * per
    out["is_afternoon"] = ((out["event_start_hour"] >= 12) & (out["event_start_hour"] < 20)).astype(float)
    out["fire_radius_km"] = np.sqrt(area_first * 10000.0 / np.pi) / 1000.0
    out["area_to_dist_ratio"] = area_first / (dist / 1000.0 + 0.1)

    out = out.replace([np.inf, -np.inf], np.nan).fillna(0.0)
    return out




def compute_c_index(time: np.ndarray, event: np.ndarray, risk: np.ndarray) -> float:
    n = len(time)
    concordant = 0.0
    comparable = 0.0
    for i in range(n):
        if event[i] != 1:
            continue
        ti = time[i]
        ri = risk[i]
        for j in range(n):
            if i == j or ti >= time[j]:
                continue
            comparable += 1.0
            if ri > risk[j]:
                concordant += 1.0
            elif ri == risk[j]:
                concordant += 0.5
    return concordant / comparable if comparable > 0 else 0.5


def compute_brier(time: np.ndarray, event: np.ndarray, prob: np.ndarray, horizon: float) -> float:
    valid = ~((event == 0) & (time < horizon))
    y = ((event == 1) & (time <= horizon)).astype(float)[valid]
    p = np.clip(prob[valid], 0.0, 1.0)
    return float(np.mean((p - y) ** 2))


def compute_hybrid_proxy(time: np.ndarray, event: np.ndarray, p24: np.ndarray, p48: np.ndarray, p72: np.ndarray) -> tuple[float, float, float, float, float]:
    risk = 0.3 * p24 + 0.4 * p48 + 0.3 * p72
    c_idx = compute_c_index(time, event, risk)
    b24 = compute_brier(time, event, p24, 24.0)
    b48 = compute_brier(time, event, p48, 48.0)
    b72 = compute_brier(time, event, p72, 72.0)
    wb = 0.3 * b24 + 0.4 * b48 + 0.3 * b72
    score = 0.3 * c_idx + 0.7 * (1.0 - wb)
    return score, c_idx, b24, b48, b72


def bagged_oof_and_predict(
    model,
    X_train: pd.DataFrame,
    y_train: np.ndarray,
    X_pred: pd.DataFrame,
    seeds=SEEDS,
    n_splits: int = N_SPLITS,
):
    n_train = len(X_train)
    n_pred = len(X_pred)
    oof = np.zeros(n_train, dtype=float)
    cnt = np.zeros(n_train, dtype=float)
    pred = np.zeros(n_pred, dtype=float)
    total_models = len(seeds) * n_splits

    for seed in seeds:
        kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
        for tr_idx, va_idx in kf.split(X_train):
            m = clone(model)
            m.fit(X_train.iloc[tr_idx], y_train[tr_idx])
            oof[va_idx] += m.predict(X_train.iloc[va_idx])
            cnt[va_idx] += 1.0
            pred += m.predict(X_pred) / total_models

    oof /= np.maximum(cnt, 1.0)
    return oof, pred


def probs_from_logtime(log_time_pred: np.ndarray) -> np.ndarray:
    p12 = norm.cdf((np.log1p(12.0) - log_time_pred) / SIGMA_12)
    p24 = norm.cdf((np.log1p(24.0) - log_time_pred) / SIGMA_24)
    p48 = norm.cdf((np.log1p(48.0) - log_time_pred) / SIGMA_48)
    return np.clip(np.column_stack([p12, p24, p48]), 0.0, 1.0)


def enforce_monotonicity(preds: np.ndarray) -> np.ndarray:
    out = np.clip(preds, 0.0, 1.0)
    for j in range(1, out.shape[1]):
        out[:, j] = np.maximum(out[:, j], out[:, j - 1])
    return out


def main():
    data_dir = locate_data_dir()
    output_path = Path("/kaggle/working/submission.csv") if Path("/kaggle/working").exists() else Path("/mnt/data/submission.csv")

    train_df = pd.read_csv(data_dir / "train.csv")
    test_df = pd.read_csv(data_dir / "test.csv")
    sample_sub = pd.read_csv(data_dir / "sample_submission.csv")

    feat_train = create_features(train_df)
    feat_test = create_features(test_df)

    feature_cols = [
        "dt_first_last_0_5h",
        "num_perimeters_0_5h",
        "low_temporal_resolution_0_5h",
        "alignment_abs",
        "alignment_cos",
        "along_track_speed",
        "cross_track_component",
        "spread_bearing_sin",
        "spread_bearing_cos",
        "spread_bearing_deg",
        "log1p_area_first",
        "area_first_ha",
        "log1p_growth",
        "log_area_ratio_0_5h",
        "radial_growth_rate_m_per_h",
        "radial_growth_m",
        "area_growth_rate_ha_per_h",
        "dist_fit_r2_0_5h",
        "dist_std_ci_0_5h",
        "dist_accel_m_per_h2",
        "dist_change_ci_0_5h",
        "dist_slope_ci_0_5h",
        "closing_speed_m_per_h",
        "log_eta",
        "eta_effective",
        "threat_score",
        "fire_urgency",
        "growth_intensity",
        "event_start_hour",
        "event_start_dayofweek",
        "event_start_month",
        "is_afternoon",
        "fire_radius_km",
        "area_to_dist_ratio",
    ]

    near_train_mask = train_df["dist_min_ci_0_5h"].values < THRESHOLD_M

    X_near_train = feat_train.loc[near_train_mask, feature_cols].copy()
    X_all_test = feat_test[feature_cols].copy()
    y_near_logtime = np.log1p(train_df.loc[near_train_mask, "time_to_hit_hours"].values)

    svr_model = Pipeline([
        ("scaler", StandardScaler()),
        ("model", SVR(C=3.0, epsilon=0.05, kernel="rbf", gamma="scale")),
    ])
    ridge_model = Pipeline([
        ("scaler", StandardScaler()),
        ("model", Ridge(alpha=3.0)),
    ])

    oof_svr, test_svr = bagged_oof_and_predict(svr_model, X_near_train, y_near_logtime, X_all_test)
    oof_ridge, test_ridge = bagged_oof_and_predict(ridge_model, X_near_train, y_near_logtime, X_all_test)

    oof_blend = BLEND_SVR * oof_svr + BLEND_RIDGE * oof_ridge
    test_blend = BLEND_SVR * test_svr + BLEND_RIDGE * test_ridge

    oof_probs = probs_from_logtime(oof_blend)
    proxy_train = np.zeros((len(train_df), 4), dtype=float)
    proxy_train[:, 3] = 1.0
    proxy_train[near_train_mask, :3] = oof_probs
    proxy_train = enforce_monotonicity(proxy_train)
    proxy_score, proxy_c, proxy_b24, proxy_b48, proxy_b72 = compute_hybrid_proxy(
        train_df["time_to_hit_hours"].values,
        train_df["event"].values,
        proxy_train[:, 1],
        proxy_train[:, 2],
        proxy_train[:, 3],
    )

    probs_12_24_48 = probs_from_logtime(test_blend)

    submission = pd.DataFrame({
        "event_id": test_df["event_id"].values,
        "prob_12h": 0.0,
        "prob_24h": 0.0,
        "prob_48h": 0.0,
        "prob_72h": 1.0,
    })

    near_test_mask = test_df["dist_min_ci_0_5h"].values < THRESHOLD_M
    submission.loc[near_test_mask, ["prob_12h", "prob_24h", "prob_48h"]] = probs_12_24_48[near_test_mask]

    submission[["prob_12h", "prob_24h", "prob_48h", "prob_72h"]] = enforce_monotonicity(
        submission[["prob_12h", "prob_24h", "prob_48h", "prob_72h"]].values
    )

    submission = sample_sub[["event_id"]].merge(submission, on="event_id", how="left")
    submission.to_csv(output_path, index=False)

    print(f"DATA_DIR      : {data_dir}")
    print(f"OUTPUT_PATH   : {output_path}")
    print(f"TRAIN_ROWS    : {len(train_df)}")
    print(f"TEST_ROWS     : {len(test_df)}")
    print(f"NEAR_TRAIN    : {int(near_train_mask.sum())}")
    print(f"NEAR_TEST     : {int(near_test_mask.sum())}")
    print(f"MONOTONIC_OK  : {bool((submission.prob_12h <= submission.prob_24h).all() and (submission.prob_24h <= submission.prob_48h).all() and (submission.prob_48h <= submission.prob_72h).all())}")
    print(f"RANGE_OK      : {bool(((submission.iloc[:, 1:] >= 0.0) & (submission.iloc[:, 1:] <= 1.0)).all().all())}")
    print(f"PROXY_HYBRID  : {proxy_score:.6f}")
    print(f"PROXY_CINDEX  : {proxy_c:.6f}")
    print(f"PROXY_B24     : {proxy_b24:.6f}")
    print(f"PROXY_B48     : {proxy_b48:.6f}")
    print(f"PROXY_B72     : {proxy_b72:.6f}")
    print(submission.head(10).to_string(index=False))


if __name__ == "__main__":
    main()


DATA_DIR      : /kaggle/input/competitions/WiDSWorldWide_GlobalDathon26
OUTPUT_PATH   : /kaggle/working/submission.csv
TRAIN_ROWS    : 221
TEST_ROWS     : 95
NEAR_TRAIN    : 69
NEAR_TEST     : 28
MONOTONIC_OK  : True
RANGE_OK      : True
PROXY_HYBRID  : 0.974882
PROXY_CINDEX  : 0.944021
PROXY_B24     : 0.022990
PROXY_B48     : 0.012486
PROXY_B72     : 0.000000
 event_id  prob_12h  prob_24h  prob_48h  prob_72h
 10662602  0.000000  0.000000  0.000000       1.0
 13353600  0.664855  0.997874  0.997874       1.0
 13942327  0.000000  0.000000  0.000000       1.0
 16112781  0.691257  0.998932  0.998932       1.0
 17132808  0.000000  0.000000  0.000000       1.0
 17445696  0.000000  0.000000  0.000000       1.0
 17599982  0.000000  0.000000  0.000000       1.0
 18750374  0.668329  0.998051  0.998051       1.0
 21365245  0.000000  0.000000  0.000000       1.0
 23634840  0.515184  0.959444  0.959444       1.0
